In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Use the official MediaWiki API to fetch parsed page HTML to avoid 403 Forbidden
api_url = "https://liquipedia.net/pubgmobile/api.php"
params = {
    "action": "parse",
    "page": "PUBG_Mobile_Global_Open/2026/Season_1/Statistics/Teams",
    "format": "json"
}
headers = {
    "User-Agent": "PMGO_Dashboard/1.0 (contact@example.com)"
}

response = requests.get(api_url, params=params, headers=headers)
print(f"API Response Status Code: {response.status_code}")

if response.status_code == 200:
    html = response.json().get("parse", {}).get("text", {}).get("*", "")
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    print(f"Found {len(tables)} tables")
else:
    print(f"Failed to fetch data: {response.text}")

API Response Status Code: 200
Found 7 tables


### Bypassing 403 Forbidden via MediaWiki API

Liquipedia uses Cloudflare protection to block automated scraping of HTML pages (returning a `403 Forbidden` error).
However, they provide an official MediaWiki API that permits data access under specific guidelines (using a custom User-Agent and respecting rate limits).
We use the API's `action=parse` action to retrieve the parsed HTML of the team statistics page, allowing us to find the tables programmatically.

In [3]:
import pandas as pd

df = pd.read_csv('../data/team_standings.csv')
print(df.shape)
df.head()

(16, 28)


,Rank,Participant,Total Points,MPe Game,Game 1 P,Game 1 K,Game 2 P,Game 2 K,Game 3 P,Game 3 K,...,Game 8 P,Game 8 K,Game 9 P,Game 9 K,Game 10 P,Game 10 K,Game 11 P,Game 11 K,Game 12 P,Game 12 K
0,1st,4Thrives,122,Game 8,10th,2,1st,9,3rd,8,...,14th,5,8th,1,13th,3,9th,1,3rd,4
1,2nd,ULF Esports,104,Game 11,3rd,5,7th,5,1st,13,...,6th,3,2nd,17,4th,16,5th,2,9th,0
2,3rd,FURIA,97,NaN,1st,16,8th,10,12th,0,...,9th,3,1st,11,7th,0,13th,3,2nd,13
3,4th,S2G Esports,83,NaN,14th,0,15th,0,2nd,3,...,4th,5,16th,0,9th,2,1st,17,6th,7
4,5th,eArena,83,NaN,11th,4,2nd,6,6th,8,...,1st,12,5th,6,8th,0,3rd,5,15th,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     str  
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     str  
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     str  
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     str  
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     str  
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     str  
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     str  
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     str  
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     str  
 19  Game 8 K      16 non-null     int64
 20  G

In [7]:
df[['Game 1 P', 'Game 2 P', 'Game 3 P']].head(10)

,Game 1 P,Game 2 P,Game 3 P
0,10th,1st,3rd
1,3rd,7th,1st
2,1st,8th,12th
3,14th,15th,2nd
4,11th,2nd,6th
5,12th,16th,15th
6,5th,6th,5th
7,16th,13th,10th
8,6th,10th,9th
9,9th,5th,7th


In [8]:
df['Rank'].head(10)

0     1st
1     2nd
2     3rd
3     4th
4     5th
5     6th
6     7th
7     8th
8     9th
9    10th
Name: Rank, dtype: str

In [8]:
import re

def clean_rank(value):
    if pd.isna(value):
        return None
    return int(re.sub(r'(st|nd|rd|th)$', '', str(value).strip()))

# Clean the overall Rank column
df['Rank'] = df['Rank'].apply(clean_rank)

# Clean every "Game X P" column
placement_cols = [col for col in df.columns if col.endswith(' P')]
for col in placement_cols:
    df[col] = df[col].apply(clean_rank)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     int64
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     int64
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     int64
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     int64
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     int64
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     int64
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     int64
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     int64
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     int64
 19  Game 8 K      16 non-null     int64
 20  G

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     int64
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     int64
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     int64
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     int64
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     int64
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     int64
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     int64
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     int64
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     int64
 19  Game 8 K      16 non-null     int64
 20  G

In [9]:
kill_cols = [col for col in df.columns if col.endswith(' K')]
df['Total Kill Points'] = df[kill_cols].sum(axis=1)
df['Total Placement Points'] = df['Total Points'] - df['Total Kill Points']

df[['Participant', 'Total Points', 'Total Kill Points', 'Total Placement Points']].sort_values('Total Points', ascending=False)

,Participant,Total Points,Total Kill Points,Total Placement Points
0,4Thrives,122,77,45
1,ULF Esports,104,70,34
2,FURIA,97,65,32
3,S2G Esports,83,53,30
4,eArena,83,56,27
5,TT Project,77,51,26
6,GOAT Team,75,49,26
7,Aurora,73,47,26
8,BOOM Esports,71,49,22
9,Team Flash,69,45,24


In [11]:
df.to_csv('../data/team_standings_clean.csv', index=False)
print("Saved!")

Saved!


In [10]:
game_rows = []

for _, row in df.iterrows():
    for game_num in range(1, 13):
        p_col = f"Game {game_num} P"
        k_col = f"Game {game_num} K"
        game_rows.append({
            "Participant": row["Participant"],
            "Game": game_num,
            "Placement": row[p_col],
            "Kill Points": row[k_col],
        })

games_df = pd.DataFrame(game_rows)
games_df.head(15)

,Participant,Game,Placement,Kill Points
0,4Thrives,1,10,2
1,4Thrives,2,1,9
2,4Thrives,3,3,8
3,4Thrives,4,16,0
4,4Thrives,5,1,16
5,4Thrives,6,1,12
6,4Thrives,7,4,16
7,4Thrives,8,14,5
8,4Thrives,9,8,1
9,4Thrives,10,13,3


In [12]:
placement_points_map = {
    1: 10, 2: 6, 3: 5, 4: 4, 5: 3, 6: 2,
    7: 1, 8: 1, 9: 0, 10: 0, 11: 0, 12: 0,
    13: 0, 14: 0, 15: 0, 16: 0
}

games_df["Placement Points"] = games_df["Placement"].map(placement_points_map)
games_df["Game Total Points"] = games_df["Placement Points"] + games_df["Kill Points"]
games_df.head(15)

,Participant,Game,Placement,Kill Points,Placement Points,Game Total Points
0,4Thrives,1,10,2,0,2
1,4Thrives,2,1,9,10,19
2,4Thrives,3,3,8,5,13
3,4Thrives,4,16,0,0,0
4,4Thrives,5,1,16,10,26
5,4Thrives,6,1,12,10,22
6,4Thrives,7,4,16,4,20
7,4Thrives,8,14,5,0,5
8,4Thrives,9,8,1,1,2
9,4Thrives,10,13,3,0,3


In [13]:
games_df = games_df.sort_values(["Participant", "Game"])
games_df["Cumulative Points"] = games_df.groupby("Participant")["Game Total Points"].cumsum()
games_df.head(15)

,Participant,Game,Placement,Kill Points,Placement Points,Game Total Points,Cumulative Points
0,4Thrives,1,10,2,0,2,2
1,4Thrives,2,1,9,10,19,21
2,4Thrives,3,3,8,5,13,34
3,4Thrives,4,16,0,0,0,34
4,4Thrives,5,1,16,10,26,60
5,4Thrives,6,1,12,10,22,82
6,4Thrives,7,4,16,4,20,102
7,4Thrives,8,14,5,0,5,107
8,4Thrives,9,8,1,1,2,109
9,4Thrives,10,13,3,0,3,112


In [14]:
check = games_df.groupby("Participant")["Game Total Points"].sum().reset_index()
check = check.merge(df[["Participant", "Total Points"]], on="Participant")
check["Difference"] = check["Total Points"] - check["Game Total Points"]
check.sort_values("Difference", ascending=False)

,Participant,Game Total Points,Total Points,Difference
0,4Thrives,122,122,0
1,721 Esports,51,51,0
2,Aurora,73,73,0
3,BOOM Esports,71,71,0
4,Bigetron,53,53,0
5,FURIA,97,97,0
6,GOAT Team,75,75,0
7,Gaming Stars,59,59,0
8,Geekay,54,54,0
9,Horaa Esports,59,59,0


In [16]:
games_df.to_csv('../data/games_long_clean.csv', index=False)
print("Saved!")

Saved!


In [18]:
players_df = pd.read_csv('../data/player_stats.csv')
print(players_df.shape)
players_df.head(10)

(64, 6)


,Player,Elims,Avg. Dmg.\nDealt,Assists,Knocks,KD\nRatio
0,Kecth,27,420,13,19,2.25
1,Ayala,22,371,12,21,1.83
2,HamsiG,22,352,7,20,1.83
3,NEOZ,21,321,7,21,1.75
4,Silenceee,20,335,15,21,1.67
5,T24OP,20,329,13,17,1.67
6,SkY,19,373,10,20,1.58
7,IQ,19,349,12,21,1.58
8,Focus,19,333,6,19,1.58
9,HUZAIFA,19,300,6,17,1.58


In [20]:
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player           64 non-null     str    
 1   Elims            64 non-null     int64  
 2   Avg. Dmg.
Dealt  64 non-null     int64  
 3   Assists          64 non-null     int64  
 4   Knocks           64 non-null     int64  
 5   KD
Ratio         64 non-null     float64
dtypes: float64(1), int64(4), str(1)
memory usage: 3.5 KB


In [22]:
players_df = pd.read_csv('../data/player_stats.csv')
players_df.info()
players_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player           64 non-null     str    
 1   Team             64 non-null     str    
 2   Elims            64 non-null     int64  
 3   Avg. Dmg.
Dealt  64 non-null     int64  
 4   Assists          64 non-null     int64  
 5   Knocks           64 non-null     int64  
 6   KD
Ratio         64 non-null     float64
dtypes: float64(1), int64(4), str(2)
memory usage: 4.7 KB


,Player,Team,Elims,Avg. Dmg.\nDealt,Assists,Knocks,KD\nRatio
0,Kecth,ULF Esports,27,420,13,19,2.25
1,Ayala,FURIA,22,371,12,21,1.83
2,HamsiG,S2G Esports,22,352,7,20,1.83
3,NEOZ,TT Project,21,321,7,21,1.75
4,Silenceee,FURIA,20,335,15,21,1.67


In [24]:
players_df.columns = players_df.columns.str.replace('\n', ' ', regex=False)
players_df.columns.tolist()

['Player', 'Team', 'Elims', 'Avg. Dmg. Dealt', 'Assists', 'Knocks', 'KD Ratio']

In [25]:
players_df.to_csv('../data/player_stats_clean.csv', index=False)
print("Saved!")

Saved!
